In [4]:
import pandas as pd
import numpy as np

In [23]:
df= pd.read_parquet("../Data/raw/all_laps.parquet")
df.head()

,year,round,session,session_folder,event,country,location,event_date,Time,Driver,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,0 days 00:39:09.674000,HAM,...,False,Mercedes,0 days 00:37:09.970000,2021-03-28 15:07:09.980,124,2.0,False,,False,False
1,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,0 days 00:41:32.386000,HAM,...,False,Mercedes,0 days 00:39:09.674000,2021-03-28 15:09:09.684,4,2.0,False,,False,False
2,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,0 days 00:44:09.042000,HAM,...,False,Mercedes,0 days 00:41:32.386000,2021-03-28 15:11:32.396,41,2.0,False,,False,False
3,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,0 days 00:45:53.974000,HAM,...,False,Mercedes,0 days 00:44:09.042000,2021-03-28 15:14:09.052,126,2.0,False,,False,False
4,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,0 days 00:47:39.113000,HAM,...,False,Mercedes,0 days 00:45:53.974000,2021-03-28 15:15:53.984,671,2.0,False,,False,False


In [14]:
# Checking the columns in each df and what they represent
df.columns

Index(['year', 'round', 'session', 'session_folder', 'event', 'country',
       'location', 'event_date', 'Time', 'Driver', 'DriverNumber', 'LapTime',
       'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time',
       'Sector2Time', 'Sector3Time', 'Sector1SessionTime',
       'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2',
       'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife',
       'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus',
       'Position', 'Deleted', 'DeletedReason', 'FastF1Generated',
       'IsAccurate'],
      dtype='str')

The columns in this df can be grouped into 6 categories:
- *Race Identification* which tells us which race and session the data belongs to.
    - The columns in this category are `year`, `round`, `session`, `session_folder`, `event`, `country`, `location`, and `event_date`.

- *Driver and team* which tells us who dreve the lap and for which team.
    - The columns in this category are `Driver`, `DriverNumber`, and `Team`.

- *Lap and Timing* which tells us how long laps and individual sectors took.
    - The columns in this category are `Time`, `LapTime`, `LapNumber`, `Sector1Time`, `Sector2Time`, `Sector3Time`.

- *Pit stops and stints* which tells us tyre stints and pit-lane activity.
    - The columns in this category are `Stint`, `PitInTime`, `PitOutTime`.

- *Tyres* which tells us tyre type, age and whether or not the tyre was fresh.
    - The columns in this category are `Compound`, `TyreLife`, `FreshTyre`

- *Track and data quality* which tells us track conditions, performance, time stamps and data reliability.
    - The columns in this category are `SpeedI1`, `SpeedI2`, `SpeedFL`, `SpeedST`, `TrackStatus`, `Position`, `IsPersonalBest`, `Deleted`, `DeletedReason`, `FastF1Generated`, `IsAccurate`, `LapStartTime`, `LapStartDate`, `Sector1SessionTime`, `Sector2SessionTime`, `Sector3SessionTime`

A few of these columns are particularly important for this project;
- `LapTime` : Will be used to understand how lap pace changes.
- `Compound` : Will identify the tyre being used (soft, hard, medium, etc).
- `TyreLife` : Will identify how old the tyres are in laps.
- `Stint` : Will help distinguish different tyre stints during a race.
- `PitInTime` and `PitOutTime` : Will be used to identify pit-stop activity.
- `TrackStatus` : Will help identify changing track conditions.
- `Position` : Will help understand the drivers race position.

The `Time` column is the timestamp of the lap whereas the `LapTime` column is the actual duration of the lap.
- `Time` column tells us how long it took for a lap to end since the session begun. We use the term session instead of race because in F1 that time could be for practice, qualifying or a race.
- `LapTime` column tells us how long the lap lasted.

In [13]:
# Inspecting the data types in each column
df.dtypes

year                            int64
round                           int64
session                           str
session_folder                    str
event                             str
country                           str
location                          str
event_date             datetime64[ns]
Time                  timedelta64[ns]
Driver                            str
DriverNumber                      str
LapTime               timedelta64[ns]
LapNumber                     float64
Stint                         float64
PitOutTime            timedelta64[ns]
PitInTime             timedelta64[ns]
Sector1Time           timedelta64[ns]
Sector2Time           timedelta64[ns]
Sector3Time           timedelta64[ns]
Sector1SessionTime    timedelta64[ns]
Sector2SessionTime    timedelta64[ns]
Sector3SessionTime    timedelta64[ns]
SpeedI1                       float64
SpeedI2                       float64
SpeedFL                       float64
SpeedST                       float64
IsPersonalBe

In the df, the data types contained in each of the column are as follows;
- int64 : `year`, `round`
- float64 : `LapNumber`, `Stint`, `SpeedI1`, `SpeedI2`, `SpeedFL`, `SpeedST`, `TyreLife`, `Position`

- string : `session`, `session_folder`, `event`, `country`, `location`, `Driver`, `DriverNumber`, `Compound`, `Team`, `TrackStatus`, `DeletedReason`

- datetime64[ns] : `event_date`, `LapStartDate`
- timedelta64[ns] : `Time`, `LapTime`, `PitOutTime`, `PitInTime`, `Sector1Time`, `Sector2Time`, `Sector3Time`, `Sector1SessionTime`, `Sector2SessionTime`, `Sector3SessionTime`,  `LapStartTime`

- bool : `FreshTyre`, `Deleted`, `FastF1Generated`, `IsAccurate`

- object : `IsPersonalBest`

In [16]:
# Checking for missing values
df.isna().sum()

year                       0
round                      0
session                    0
session_folder             0
event                      0
country                    0
location                   0
event_date                 0
Time                       0
Driver                     0
DriverNumber               0
LapTime                  526
LapNumber                  0
Stint                    354
PitOutTime            120754
PitInTime             120731
Sector1Time             2660
Sector2Time              202
Sector3Time              319
Sector1SessionTime      2931
Sector2SessionTime       202
Sector3SessionTime       319
SpeedI1                  161
SpeedI2                 1048
SpeedFL                 4473
SpeedST                  217
IsPersonalBest           113
Compound                   0
TyreLife                 989
FreshTyre                  0
Team                       0
LapStartTime               0
LapStartDate            1057
TrackStatus                0
Position      

In the df, the only columns with missing values are;
- `LapTime` with `526`
- `Stint` with `354` 
- `PitOutTime` with `120,754` : Most laps are not pit out laps so a high number of missing values is expected.
- `PitInTime` with `120,731` : Most laps are not pit in lanes so a high number of missing values is expected.
- `Sector1Time` with `2,660` : 
- `Sector2Time` with `202`
- `Sector3Time` with `319`
- `Sector1SessionTime` with `2,931`
- `Sector2SessionTime` with `202`
- `Sector3SessionTime` with `319`
- `SpeedI1` with `161`
- `SpeedI2` with `1,048`
- `SpeedFL` with `4,473`
- `SpeedST` with `217`
- `IsPersonalBest` with `113`
- `TyreLife` with `989`
- `LapStartDate` with `1,057`
- `Position` with `160`
- `DeletedReason` with `113`

All the other columns have no missing values.

In [21]:
# Checking for duplicates
df.duplicated().sum()

0         False
1         False
2         False
3         False
4         False
          ...  
125041    False
125042    False
125043    False
125044    False
125045    False
Length: 125046, dtype: bool

There are not duplicate rows in the entire df.

In order to predict whether a driver should pit on a certain lap, we need to know what compound is currently in use, how old the tyres are, how the lap time changes as the tyres age, and how long the driver has been on that particular stint.

In [44]:
# Inspecting these tyre related columns.
df[["Compound", "TyreLife", "FreshTyre", "Stint"]].head(10)

,Compound,TyreLife,FreshTyre,Stint
0,MEDIUM,4.0,False,1.0
1,MEDIUM,5.0,False,1.0
2,MEDIUM,6.0,False,1.0
3,MEDIUM,7.0,False,1.0
4,MEDIUM,8.0,False,1.0
5,MEDIUM,9.0,False,1.0
6,MEDIUM,10.0,False,1.0
7,MEDIUM,11.0,False,1.0
8,MEDIUM,12.0,False,1.0
9,MEDIUM,13.0,False,1.0


A *stint* is a continuous period during which a driver uses the same set of tyres. When a driver pits and changes tyres, a new stint begins.

In [54]:
# Inspecting unique tyre compounds
df["Compound"].value_counts()

Compound
HARD            54845
MEDIUM          45511
SOFT            15682
INTERMEDIATE     7675
WET               493
None              426
nan               354
UNKNOWN            60
Name: count, dtype: int64

There are more than 3 dry weather compounds.

There are intermdiate and wet tyres too.

We should not treat every lap as if it happened under dry conditions.

For the strategy model, weather and track conditions will matter when estimate tyre perfomance.

In [55]:
# Evaluating missing values in Compound.
df[df["Compound"].isna() | df["Compound"].isin(["None", "nan"])]["Compound"].value_counts(dropna=False)

Compound
None    426
nan     354
Name: count, dtype: int64

The `Compound` column contains two values that represent missing tyre compounds: 
- `None` occurs 426 
- `nan` occurs 354

These are stored as string values rather than actual NaN values. This means they are not automatically detected by missing-value checks such as `.isna()`.

Before stadardizing the missing values, we will check their distribution across seasons and sessions.

In [56]:
# Examining distribution of the missing values in Compound.
columns= ["year", "session", "event", "Driver", "LapNumber", "Compound"]

df[df["Compound"].isin(["None", "nan"])][columns].groupby(["year", "session", 'Compound']).size().unstack(fill_value=0)

,Compound,None,nan
year,session,,
2022,Race,334,0
2023,Race,35,0
2025,Race,57,354


The missing values are only found in Race sessions across the 2022, 2023 and 2025 seasons.

The distribution is uneven: 
- `None` occurs in 2022, 2023 and 2025
- `nan` occurs only in 2025

This may indicate differences in how missing compound information was recorded across seasons.

In [61]:
# Getting 10 records for each missing value and trying to understand why compound was not recorded.
columns = ["year", "event", "Driver", "LapNumber", "LapTime", "Compound", "Stint", "TyreLife", "FreshTyre", "PitInTime", "PitOutTime", "TrackStatus", "Deleted", "IsAccurate"]

df[df["Compound"].isin(["None", "nan"])][columns].sort_values(["year", "event", "Driver", "LapNumber"]).groupby("Compound").head(10)

,year,event,Driver,LapNumber,LapTime,Compound,Stint,TyreLife,FreshTyre,PitInTime,PitOutTime,TrackStatus,Deleted,IsAccurate
34872,2022,Austrian Grand Prix,ALB,43.0,0 days 00:01:10.395000,None,3.0,NaN,False,NaT,NaT,1,False,True
34873,2022,Austrian Grand Prix,ALB,44.0,0 days 00:01:10.320000,None,3.0,NaN,False,NaT,NaT,1,False,True
34874,2022,Austrian Grand Prix,ALB,45.0,0 days 00:01:10.904000,None,3.0,NaN,False,NaT,NaT,1,False,True
34875,2022,Austrian Grand Prix,ALB,46.0,0 days 00:01:10.779000,None,3.0,NaN,False,NaT,NaT,1,False,True
34876,2022,Austrian Grand Prix,ALB,47.0,0 days 00:01:10.802000,None,3.0,NaN,False,NaT,NaT,1,False,True
34877,2022,Austrian Grand Prix,ALB,48.0,0 days 00:01:10.605000,None,3.0,NaN,False,NaT,NaT,1,False,True
34878,2022,Austrian Grand Prix,ALB,49.0,0 days 00:01:10.827000,None,3.0,NaN,False,NaT,NaT,1,False,True
34879,2022,Austrian Grand Prix,ALB,50.0,0 days 00:01:10.466000,None,3.0,NaN,False,NaT,NaT,1,False,True
34880,2022,Austrian Grand Prix,ALB,51.0,0 days 00:01:10.474000,None,3.0,NaN,False,NaT,NaT,1,False,True
34881,2022,Austrian Grand Prix,ALB,52.0,0 days 00:01:13.466000,None,3.0,NaN,False,NaT,NaT,1,False,True


Records with missing compounds can contain vialid lap times and other useful information.

However, some records also have missing stint and tyre-life information.

In [63]:
# Sessions in each event classified as accurate or inaccurate.
df[df["Compound"].isin(["None", "nan"])].groupby(["year", "event", "Compound", "IsAccurate"]).size().unstack("IsAccurate", fill_value=0)

IsAccurate                         False  True 
year event               Compound              
2022 Austrian Grand Prix None         38    261
     Monaco Grand Prix   None          1     34
2023 Canadian Grand Prix None          0     35
2025 Belgian Grand Prix  None          4     53
     Miami Grand Prix    nan          45    309

780 records contain the strings `"None"` or `"nan"` in the `Compound` column.

These values represent missing values. 

Most of the affected records (692) are marked as accurate by the source data.

Both strings will be converted into pandas missing values.

No rows will be deleted because the records may still contain useful lap-time and race information.

In [65]:
df["Compound"] = df["Compound"].replace(["None", "nan"], np.nan)

df["Compound"].value_counts(dropna=False)

Compound
HARD            54845
MEDIUM          45511
SOFT            15682
INTERMEDIATE     7675
NaN               780
WET               493
UNKNOWN            60
Name: count, dtype: int64

In [66]:
# Investigating distribution of 'UNKNOWN'. 
df[df["Compound"] == "UNKNOWN"].groupby(["year", "session", "event"]).size().sort_values(ascending=False)

year  session  event             
2021  Race     Belgian Grand Prix    60
dtype: int64

All 60 records labelled `UNKNOWN` belong to the 2021 Belgian Grand Prix. 

In [67]:
df[df["Compound"] == "UNKNOWN"][columns].sort_values(["Driver", "LapNumber"]).head(15)

,year,event,Driver,LapNumber,LapTime,Compound,Stint,TyreLife,FreshTyre,PitInTime,PitOutTime,TrackStatus,Deleted,IsAccurate
12678,2021,Belgian Grand Prix,ALO,1.0,0 days 00:03:47.661000,UNKNOWN,1.0,1.0,False,NaT,0 days 04:16:41.190000,1,False,False
12679,2021,Belgian Grand Prix,ALO,2.0,0 days 00:03:18.069000,UNKNOWN,1.0,2.0,False,NaT,NaT,1,False,True
12680,2021,Belgian Grand Prix,ALO,3.0,0 days 00:03:31.617000,UNKNOWN,1.0,3.0,False,0 days 04:26:26.782000,NaT,15,False,False
12681,2021,Belgian Grand Prix,BOT,1.0,0 days 00:03:49.485000,UNKNOWN,1.0,1.0,False,NaT,0 days 04:16:43.184000,1,False,False
12682,2021,Belgian Grand Prix,BOT,2.0,0 days 00:03:18.036000,UNKNOWN,1.0,2.0,False,NaT,NaT,1,False,True
12683,2021,Belgian Grand Prix,BOT,3.0,0 days 00:03:31.763000,UNKNOWN,1.0,3.0,False,0 days 04:26:28.946000,NaT,15,False,False
12663,2021,Belgian Grand Prix,GAS,1.0,0 days 00:03:37.248000,UNKNOWN,1.0,1.0,False,NaT,0 days 04:16:22.073000,1,False,False
12664,2021,Belgian Grand Prix,GAS,2.0,0 days 00:03:19.284000,UNKNOWN,1.0,2.0,False,NaT,NaT,1,False,True
12665,2021,Belgian Grand Prix,GAS,3.0,0 days 00:03:29.238000,UNKNOWN,1.0,3.0,False,0 days 04:26:17.408000,NaT,15,False,False
12684,2021,Belgian Grand Prix,GIO,1.0,0 days 00:03:51.234000,UNKNOWN,1.0,1.0,False,NaT,0 days 04:16:47.072000,1,False,False


The 60 records containing `UNKNOWN` all belong to the 2021 Belgian Grand Prix. 

Since the actual tyre compounds cannot be established from these records alone, `UNKNOWN` will be treated as missing information rather than assigning an assumed compound.

In [68]:
df["Compound"] = df["Compound"].replace("UNKNOWN", np.nan)

df["Compound"].value_counts(dropna=False)

Compound
HARD            54845
MEDIUM          45511
SOFT            15682
INTERMEDIATE     7675
NaN               840
WET               493
Name: count, dtype: int64

The `IsPersonalBest` column indicates whether a lap was a driver's personal best. 

It has 113 missing values.

In [69]:
df["IsPersonalBest"].value_counts(dropna=False)

IsPersonalBest
False    98551
True     26382
None       113
Name: count, dtype: int64

The `IsPersonalBest` column contains Boolean values and 113 missing entries represented by the string "None".

These entries will be converted to NaN while maintaing existing True/False values.

In [70]:
df["IsPersonalBest"] = df["IsPersonalBest"].replace("None", np.nan)

df["IsPersonalBest"].value_counts(dropna=False)

IsPersonalBest
False    98551
True     26382
None       113
Name: count, dtype: int64

In [71]:
df["IsPersonalBest"].isna().sum()

np.int64(113)

LapTime is an important variable in the project because it measures how long a driver takes to complete a lap.

There are 526 missing values.

In [72]:
df[df["LapTime"].isna()].groupby(["year", "session"]).size()

year  session
2021  Race       142
2022  Race       103
2023  Race       167
2024  Race        79
2025  Race        35
dtype: int64

In [73]:
df["session"].unique()

<ArrowStringArray>
['Race']
Length: 1, dtype: str

Since the only session is race, figuring out the distribution is kind of redundant.

In [74]:
df[df["LapTime"].isna()][columns].head(15)

,year,event,Driver,LapNumber,LapTime,Compound,Stint,TyreLife,FreshTyre,PitInTime,PitOutTime,TrackStatus,Deleted,IsAccurate
1026,2021,Bahrain Grand Prix,MAZ,1.0,NaT,MEDIUM,1.0,1.0,True,NaT,NaT,124,False,False
1059,2021,Emilia Romagna Grand Prix,GAS,33.0,NaT,MEDIUM,4.0,6.0,False,0 days 01:55:10.239000,0 days 01:53:32.081000,51,False,False
1123,2021,Emilia Romagna Grand Prix,PER,34.0,NaT,SOFT,3.0,3.0,False,NaT,0 days 01:55:32.835000,1,False,False
1185,2021,Emilia Romagna Grand Prix,ALO,33.0,NaT,MEDIUM,3.0,4.0,False,0 days 01:55:05.161000,0 days 01:53:25.781000,51,False,False
1249,2021,Emilia Romagna Grand Prix,LEC,34.0,NaT,MEDIUM,3.0,6.0,False,NaT,0 days 01:55:18.369000,1,False,False
1311,2021,Emilia Romagna Grand Prix,STR,33.0,NaT,MEDIUM,3.0,5.0,False,0 days 01:54:37.429000,0 days 01:53:06.264000,51,False,False
1374,2021,Emilia Romagna Grand Prix,TSU,33.0,NaT,SOFT,3.0,1.0,True,0 days 01:54:53.306000,0 days 01:53:16.283000,51,False,False
1438,2021,Emilia Romagna Grand Prix,RIC,34.0,NaT,SOFT,3.0,3.0,False,NaT,0 days 01:55:41.188000,1,False,False
1500,2021,Emilia Romagna Grand Prix,OCO,33.0,NaT,MEDIUM,5.0,5.0,False,0 days 01:55:07.285000,0 days 01:53:27.319000,51,False,False
1564,2021,Emilia Romagna Grand Prix,VER,34.0,NaT,MEDIUM,3.0,6.0,False,NaT,0 days 01:55:13.733000,1,False,False


The sample of records with missing LapTime values contains several laps marked as inaccurate. 

We will explore the relationship between missing lap times and IsAccurate.

In [75]:
df[df["LapTime"].isna()]["IsAccurate"].value_counts(dropna=False)

IsAccurate
False    526
Name: count, dtype: int64

Every missing lap time belongs to a lap that FastF1 has already flagged as inaccurate.

They may still contain useful information for pit-stop and race-event analysis so they will not be deleted. At least not yet.

The Deleted column indicates whether a lap was deleted. 

We will inspect its values to understand how many records are affected and whether deleted laps require special treatment during modelling.

In [77]:
df["Deleted"].value_counts(dropna=False)

Deleted
False    123583
True       1463
Name: count, dtype: int64

We have 1,463 deleted laps out of 125,046 records (approximately 1.17%).

These are important to distinguish from inaccurate laps:
- Deleted = True means the lap was deleted.
- IsAccurate = False means the timing data is flagged as inaccurate.

Compare the Deleted and IsAccurate columns to determine whether deleted laps are consistently marked as inaccurate and whether any non-deleted laps also have inaccurate timing data.

In [78]:
pd.crosstab(df["Deleted"], df["IsAccurate"])

IsAccurate,False,True
Deleted,,
False,16625,106958
True,85,1378


1,378 deleted laps are still marked accurate.
Only 85 deleted laps are marked inaccurate.
There are also 16,625 non-deleted laps marked inaccurate.

This indicates that `Deleted` and `IsAccurate` measure different things.

These records will be preserved and filters will be applied when modelling lap times in order to avoid losing potentially useful race information.

We will investigate the `DeletedReason` column to understand why certain laps were deleted and whether specific reasons require special treatment.

In [82]:
len(df["DeletedReason"].unique())

691

The `DeletedReason` column contains many distinct entries.

Rather than examining every value at once, we will focus on records where `Deleted` is *True* to understand the reasons associated with deleted laps.

In [83]:
df[df["Deleted"] == True]["DeletedReason"].value_counts(dropna=False).head(15)

DeletedReason
TRACK LIMITS AT TURN 10               61
TRACK LIMITS AT TURN 10 (NEXT LAP)    43
TRACK LIMITS AT TURN 9                38
TRACK LIMITS AT TURN 2                19
TRACK LIMITS AT TURN 4                15
TRACK LIMITS AT TURN 1                15
TRACK LIMITS AT TURN 16               15
TRACK LIMITS AT TURN 11               14
TRACK LIMITS AT TURN 19               10
TRACK LIMITS AT TURN 1 LAP 5           9
TRACK LIMITS AT TURN 9 LAP 14          8
TRACK LIMITS AT TURN 10 LAP 13         7
TRACK LIMITS AT TURN 15                6
TRACK LIMITS AT TURN 10 LAP 8          6
TRACK LIMITS AT TURN 4 LAP 7           6
Name: count, dtype: int64

Most of the deleted laps are associated with track violations.

A deleted lap may have a perfectly recorded lap time, but that time shouldn't necessarily be treated as a valid performance measurement.

Observations:
- The most frequent reason is *TRACK LIMITS AT TURN 10 (61 records)*.
- Other reasons identify specific corners and sometimes the lap on which the violation occurred.
- The reasons are descriptive, not a small set of standard categories.

The values in `DeletedReson` will be preserved as they are.

`Deleted` will be used to identify laps excluded from valid lap-time modelling.

Deleted laps might contain recorded lap times even when those times are not valid for performance analysis. 
We will compare `Deleted` with `LapTime` to understand how these conditions overlap.

In [85]:
pd.crosstab(df["Deleted"], df["LapTime"].isna())

LapTime,False,True
Deleted,,
False,123057,526
True,1463,0


All 526 missing lap times belong to non-deleted laps.

All 1,463 deleted laps actually have recorded lap times.

This reinforces the decision that `Deleted` and `IsAccurate` must be treated separately. 
A lap can have a recorded time but still be deleted.

For the lap-time model, we'll need to exclude deleted laps, inaccurate laps and records without lap times.


TyreLife indicates how many laps a tyre has completed, while Stint identifies the tyre usage period.
Both variables are important for modelling tyre degradation. 

In [86]:
df[["Stint", "TyreLife"]].isna().sum()

Stint       354
TyreLife    989
dtype: int64

- `Stint`: 354 missing values (0.28%).
- `TyreLife`: 989 missing values (0.79%).

`Stint` and `TyreLife` are closely related variables so their missing values will be compared to determine whether they tend to be missing in the same records.

In [87]:
pd.crosstab(df["Stint"].isna(), df["TyreLife"].isna())

TyreLife,False,True
Stint,,
False,124057,635
True,0,354


- All 354 rows with missing `Stint` also have missing `TyreLife`.
- Another 635 rows have a known `Stint` but missing `TyreLife`.
- There are no rows with missing `Stint` but a recorded `TyreLife`.

This suggests `TyreLife` can be unavailable even when the stint is known.

We will check how missing stint information is distributed across years and events to find out whether or not the missing values are associated with particular races.

In [88]:
df[df["Stint"].isna()].groupby(["year", "event"]).size().sort_values(ascending=False)

year  event           
2025  Miami Grand Prix    354
dtype: int64

All 354 missing `Stint` values come from the 2025 Miami Grand Prix. Those same 354 records also have missing `TyreLife`.

We identified 354 records from the 2025 Miami Grand Prix where both `Stint` and `TyreLife` are missing. 

In [90]:
# Checking the remaining records where Stint is available but TyreLife is missing.
df[df["Stint"].notna() & df["TyreLife"].isna()].groupby(["year", "event"]).size().sort_values(ascending=False)

year  event                
2022  Austrian Grand Prix      359
      Monaco Grand Prix        161
2025  Belgian Grand Prix        75
2023  Canadian Grand Prix       35
2025  Australian Grand Prix      5
dtype: int64

The remaining 635 missing `TyreLife` values are concentrated in five events.

The records will be preserved and the `TyreLife` values will be left as they are.

`LapStartDate` records the calendar date and time associated with the start of a lap. 

We will check whether missing values coincide with missing lap times or inaccurate laps before deciding how to handle them.

In [91]:
pd.crosstab(df["LapStartDate"].isna(),df["IsAccurate"])

IsAccurate,False,True
LapStartDate,,
False,16531,107458
True,179,878


We have 1,057 missing `LapStartDate` values:
- 179 belong to laps marked inaccurate.
- 878 belong to laps marked accurate.

So missing LapStartDate does not mean a lap is inaccurate.

The records will be preserved and filterd whenever an analysis requires a timestamp.


Sector times record how long a driver takes to complete each of the three sectors of a lap. 
We will check missing values across all three sectors and compare them with missing lap times and lap accuracy.

In [92]:
sector_columns = ["Sector1Time", "Sector2Time", "Sector3Time"]

df[sector_columns].isna().sum()

Sector1Time    2660
Sector2Time     202
Sector3Time     319
dtype: int64

`Sector1Time` has significantly more missing values than the other two sectors, but these counts may overlap.

We will check whether missing sector times are associated with missing lap times or laps marked inaccurate.

In [93]:
sector_missing = df[sector_columns].isna().any(axis=1)

pd.crosstab(sector_missing, [df["IsAccurate"], df["LapTime"].isna()])

IsAccurate  False         True 
LapTime     False True    False
row_0                          
False       13938     5  108336
True         2246   521       0

2,767 rows have at least one missing sector time, and every one of them is marked inaccurate.

We don't need to impute anything since we can exclude inaccurate laps.

We will check whether missing sector session timestamps are associated with inaccurate laps or missing lap times.

In [94]:
sector_session_columns = ["Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime"]

sector_session_missing = df[sector_session_columns].isna().any(axis=1)

pd.crosstab(sector_session_missing, [df["IsAccurate"], df["LapTime"].isna()])

IsAccurate  False         True 
LapTime     False True    False
row_0                          
False       13927     5  108076
True         2257   521     260

- Sector times: Every row missing at least one sector time was marked inaccurate.
- Sector session timestamps: 3,038 rows have at least one missing timestamp. Out off these, 260 are marked accurate and have valid lap times.

Those 260 records show that missing sector session timestamps do not necessarily make a lap unusable.

We will now check missing values across the four speed measurement columns and compare them with lap accuracy and lap time availability. This will help determine whether missing speed measurements affect otherwise usable laps.

In [95]:
speed_columns = ["SpeedI1", "SpeedI2", "SpeedFL", "SpeedST"]

speed_missing = df[speed_columns].isna().any(axis=1)

pd.crosstab(speed_missing.rename("AnySpeedMissing"), [df["IsAccurate"], df["LapTime"].isna()])

IsAccurate       False         True 
LapTime          False True    False
AnySpeedMissing                     
False            11886   231  107576
True              4298   295     760

Missing speed measurements do not necessarily mean a lap is unusable.

760 accurate laps with valid lap times have at least one missing speed measurement.

We will preserve all records and leave missing speed measurements unchanged. We shouldn't exclude otherwise usable laps just because one telemetry measurement is unavailable.

`Position` represents a driver's running position during a lap. We will compare missing position values with lap accuracy and lap time availability to determine whether they affect otherwise usable records.

In [96]:
pd.crosstab(df["Position"].isna(), [df["IsAccurate"], df["LapTime"].isna()])

IsAccurate  False         True 
LapTime     False True    False
Position                       
False       16184   366  108336
True            0   160       0

All 160 missing Position values belong to laps that:
- Are marked IsAccurate=False.
- Have missing LapTime.

There are no accurate laps with missing positions.

We will preserve these 160 records and leave Position missing. They won't affect our core analysis of valid lap times.

`DeletedReason` describes why a lap was deleted. We will check whether missing reasons occur only on laps that were not marked as deleted.

In [97]:
pd.crosstab(df["Deleted"], df["DeletedReason"].isna())

DeletedReason,False,True
Deleted,,
False,123470,113
True,1463,0


All 113 missing DeletedReason values belong to laps that were not deleted. Every deleted lap has a recorded reason.

All missing values across the relevant columns have been checked. The approach used has been to preserve records, avoid unjustified imputation, and leave naturally absent values alone.

In [99]:
print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

Dataset shape: (125046, 39)
Duplicate rows: 0


In [101]:
df.isna().sum()[df.isna().sum() > 0]


LapTime                  526
Stint                    354
PitOutTime            120754
PitInTime             120731
Sector1Time             2660
Sector2Time              202
Sector3Time              319
Sector1SessionTime      2931
Sector2SessionTime       202
Sector3SessionTime       319
SpeedI1                  161
SpeedI2                 1048
SpeedFL                 4473
SpeedST                  217
IsPersonalBest           113
Compound                 840
TyreLife                 989
LapStartDate            1057
Position                 160
DeletedReason            113
dtype: int64

In [103]:
df["Compound"].value_counts(dropna=False)

Compound
HARD            54845
MEDIUM          45511
SOFT            15682
INTERMEDIATE     7675
NaN               840
WET               493
Name: count, dtype: int64

The cleaned dataset will be saved in the processed data directory, leaving the original raw dataset untouched.

In [104]:
output_path = "../Data/processed/all_laps_cleaned.parquet"

df.to_parquet(output_path, index=False)

# Verify the saved dataset
df_check = pd.read_parquet(output_path)

print("Saved successfully:", output_path)
print("Saved dataset shape:", df_check.shape)

Saved successfully: ../Data/processed/all_laps_cleaned.parquet
Saved dataset shape: (125046, 39)
